# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

## Resolución

Mismos datos y misma preparación que en los labs de feature engineering y de ensembles. El mejor modelo hasta ahora fue el **Gradient Boosting** con árboles de profundidad 3 (78,4 % de exactitud en test). Aquí le buscamos mejores hiperparámetros con las dos técnicas de la lección 7.4:

- **Grid Search** (`GridSearchCV`): prueba **todas** las combinaciones de una rejilla de valores.
- **Random Search** (`RandomizedSearchCV`): prueba solo unas cuantas combinaciones elegidas al azar. Lo usamos con el segundo mejor modelo, el Random Forest.

Las dos usan **validación cruzada** (`cv=5`): parten el entrenamiento en 5 trozos, entrenan con 4 y miden con el quinto, cinco veces. Así se elige con el entrenamiento y el test queda intacto para la nota final.

> **Resolução (português).** Os mesmos dados e a mesma preparação dos labs anteriores. O melhor modelo até agora foi o **Gradient Boosting** com árvores de profundidade 3 (78,4 % de exactidão no teste). Procuramos melhores hiperparâmetros com as duas técnicas da aula 7.4: **Grid Search** (prova todas as combinações de uma grelha) e **Random Search** (prova só algumas, ao acaso), esta com o segundo melhor modelo, o Random Forest. As duas usam **validação cruzada** (`cv=5`): dividem o treino em 5 partes, treinam com 4 e medem com a quinta, cinco vezes. Assim escolhe-se com o treino e o teste fica intacto para a nota final.

**Antes de empezar: instalar las librerías (solo la primera vez)**

`%pip install pandas numpy scikit-learn` instala lo que usa este notebook **en el mismo Python que lo ejecuta**. Si aparece `ModuleNotFoundError` (por ejemplo, `No module named 'pandas'`), es que ese Python no las tiene. Después: **Restart** y **Run All**. Si ya están instaladas, solo lo comprueba.

> **Antes de começar (português).** Esta célula instala as bibliotecas no mesmo Python que corre o notebook. Se aparecer `ModuleNotFoundError`, corre-a e depois faz **Restart** e **Run All**.

In [ ]:
%pip install pandas numpy scikit-learn

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


**Cómo funciona**

1. Limpieza y feature engineering del lab de feature engineering: `dropna`, cubierta de `Cabin`, fuera `PassengerId` y `Name`, objetivo aparte y `pd.get_dummies` con `drop_first=True`.
2. Separa el test (20 %, `random_state=0`) **antes** de escalar y seleccionar, para que nada aprenda del test.
3. Normaliza con `MinMaxScaler` ajustado solo con el entrenamiento y vuelve a `pd.DataFrame` con los nombres de las columnas.
4. Selección: se queda con las variables cuya correlación con el objetivo pasa de 0,05 en valor absoluto (lección 7.2).

**Por qué así:** el mismo preprocesado que en los labs anteriores permite comparar: si la exactitud sube, es por los hiperparámetros y no por haber cambiado los datos.

**Herramientas:** `dropna`, `.str.split`, `pd.get_dummies`, `train_test_split`, `MinMaxScaler`, `corrwith`, `np.abs`.

> **Como funciona (português)**
>
> 1. Limpeza e feature engineering do lab de feature engineering: `dropna`, coberta da `Cabin`, fora `PassengerId` e `Name`, objectivo à parte e `pd.get_dummies` com `drop_first=True`.
> 2. Separa o teste (20 %, `random_state=0`) **antes** de escalar e seleccionar, para nada aprender com o teste.
> 3. Normaliza com o `MinMaxScaler` ajustado só com o treino e volta a `pd.DataFrame` com os nomes das colunas.
> 4. Selecção: fica com as variáveis cuja correlação com o objectivo passa de 0,05 em valor absoluto (aula 7.2).
>
> **Porquê assim:** o mesmo pré-processamento dos labs anteriores permite comparar: se a exactidão sobe, é pelos hiperparâmetros e não por mudar os dados.

In [3]:
spaceship = spaceship.dropna()
spaceship["Cabin"] = spaceship["Cabin"].str.split("/").str[0]
spaceship = spaceship.drop(columns=["PassengerId", "Name"])

features = spaceship.drop(columns=["Transported"])
target = spaceship["Transported"]
columnas_texto = list(features.select_dtypes(exclude="number").columns)
features = pd.get_dummies(features, columns=columnas_texto, drop_first=True, dtype=int)

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=0)

normalizer = MinMaxScaler()
normalizer.fit(X_train)
X_train_norm = pd.DataFrame(normalizer.transform(X_train), columns=X_train.columns)
X_test_norm = pd.DataFrame(normalizer.transform(X_test), columns=X_test.columns)

corr_objetivo = np.abs(X_train_norm.corrwith(y_train.astype(int).reset_index(drop=True)))
seleccionadas = list(corr_objetivo[corr_objetivo > 0.05].index)
X_train_sel = X_train_norm[seleccionadas]
X_test_sel = X_test_norm[seleccionadas]
print("Entrenamiento:", X_train_sel.shape, "| Test:", X_test_sel.shape)
print("Variables:", seleccionadas)

Entrenamiento: (5284, 12) | Test: (1322, 12)
Variables: ['Age', 'RoomService', 'FoodCourt', 'Spa', 'VRDeck', 'HomePlanet_Europa', 'CryoSleep_True', 'Cabin_B', 'Cabin_C', 'Cabin_E', 'Cabin_F', 'Destination_TRAPPIST-1e']


**➜ Qué nos llevamos:** 6.606 pasajeros, 12 variables seleccionadas; 5.284 para entrenar y 1.322 para el test, igual que en los labs anteriores.

> **➜ O que tiramos:** 6.606 passageiros, 12 variáveis seleccionadas; 5.284 para treinar e 1.322 para o teste, como nos labs anteriores.

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

**Cómo funciona**

1. Entrena el mejor modelo del lab de ensembles con sus valores de entonces: `GradientBoostingClassifier(max_depth=3, n_estimators=100)`.
2. Es el punto de partida: lo que mejore el Grid Search se mide contra él.

**Por qué así:** sin una referencia no se puede decir si el ajuste sirve de algo.

**Herramientas:** `GradientBoostingClassifier`, `fit`.

> **Como funciona (português)**
>
> 1. Treina o melhor modelo do lab de ensembles com os valores de então: `GradientBoostingClassifier(max_depth=3, n_estimators=100)`.
> 2. É o ponto de partida: o que o Grid Search melhorar mede-se contra ele.
>
> **Porquê assim:** sem uma referência não se pode dizer se o ajuste serve para alguma coisa.

In [4]:
gb_base = GradientBoostingClassifier(max_depth=3, n_estimators=100, random_state=0)
gb_base.fit(X_train_sel, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (imp

- Evaluate your model

In [5]:
print("Gradient Boosting sin ajustar")
print("Exactitud en entrenamiento:", round(gb_base.score(X_train_sel, y_train), 3))
print("Exactitud en test:", round(gb_base.score(X_test_sel, y_test), 3))

Gradient Boosting sin ajustar
Exactitud en entrenamiento: 0.813
Exactitud en test: 0.784


**➜ Qué nos llevamos:** 78,4 % en test y 81 % en entrenamiento: poco sobreajuste. Es la nota que hay que superar.

> **➜ O que tiramos:** 78,4 % no teste e 81 % no treino: pouco sobreajuste. É a nota a superar.

**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

**Cómo funciona**

1. Define la rejilla: un diccionario con cada hiperparámetro y la lista de valores que queremos probar.
2. `n_estimators`: cuántos árboles (100, 200, 300). `max_depth`: profundidad de cada árbol (2, 3, 4). `learning_rate`: cuánto corrige cada árbol los errores del anterior (0,05 y 0,1).
3. 3 × 3 × 2 = 18 combinaciones; con `cv=5`, 90 entrenamientos.

**Por qué así:** son los hiperparámetros que más cambian un Gradient Boosting. Elegimos valores alrededor de los que ya funcionaban (100 árboles, profundidad 3) para no gastar tiempo en combinaciones absurdas.

**Herramientas:** un diccionario de listas, como en la lección (`grid = {...}`). En la lección el modelo era un AdaBoost con un árbol dentro y por eso los nombres llevaban `estimator__`; aquí el Gradient Boosting crea sus propios árboles y los hiperparámetros van sin prefijo.

> **Como funciona (português)**
>
> 1. Define a grelha: um dicionário com cada hiperparâmetro e a lista de valores a provar.
> 2. `n_estimators`: quantas árvores (100, 200, 300). `max_depth`: profundidade de cada árvore (2, 3, 4). `learning_rate`: quanto cada árvore corrige os erros da anterior (0,05 e 0,1).
> 3. 3 × 3 × 2 = 18 combinações; com `cv=5`, 90 treinos.
>
> **Porquê assim:** são os hiperparâmetros que mais mudam um Gradient Boosting. Escolhemos valores à volta dos que já funcionavam, para não gastar tempo em combinações absurdas.

In [6]:
grid = {"n_estimators": [100, 200, 300],
        "max_depth": [2, 3, 4],
        "learning_rate": [0.05, 0.1]}

- Run Grid Search

**Cómo funciona**

1. Crea el `GridSearchCV` con el modelo, la rejilla y `cv=5`.
2. `fit` entrena las 18 combinaciones con validación cruzada (tarda alrededor de un minuto).
3. `best_params_` dice la mejor combinación y `best_score_` su exactitud media en las 5 validaciones.

**Por qué así:** la validación cruzada mide cada combinación cinco veces con trozos distintos del entrenamiento: la elección no depende de la suerte de una sola partición y el test sigue sin tocarse.

**Herramientas:** `GridSearchCV(estimator=..., param_grid=..., cv=5)`, `fit`, `best_params_`, `best_score_` (lección 7.4).

> **Como funciona (português)**
>
> 1. Cria o `GridSearchCV` com o modelo, a grelha e `cv=5`.
> 2. O `fit` treina as 18 combinações com validação cruzada (cerca de um minuto).
> 3. `best_params_` dá a melhor combinação e `best_score_` a exactidão média nas 5 validações.
>
> **Porquê assim:** a validação cruzada mede cada combinação cinco vezes com partes diferentes do treino: a escolha não depende da sorte de uma só divisão e o teste continua intacto.

In [7]:
gb = GradientBoostingClassifier(random_state=0)
model = GridSearchCV(estimator=gb, param_grid=grid, cv=5)
model.fit(X_train_sel, y_train)

print("Mejores hiperparámetros:", model.best_params_)
print("Exactitud media en validación cruzada:", round(model.best_score_, 3))

Mejores hiperparámetros: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
Exactitud media en validación cruzada: 0.8


- Evaluate your model

**Cómo funciona**

1. Recupera el mejor modelo con `best_estimator_` (ya está reentrenado con todo el entrenamiento).
2. Mide su exactitud en el test, que no ha visto en ningún momento del ajuste.

**Por qué así:** el test es la nota honesta: la validación cruzada eligió el modelo, el test dice cómo funciona con datos nuevos.

**Herramientas:** `best_estimator_`, `score`.

> **Como funciona (português)**
>
> 1. Recupera o melhor modelo com `best_estimator_` (já retreinado com todo o treino).
> 2. Mede a exactidão no teste, que não viu em nenhum momento do ajuste.
>
> **Porquê assim:** o teste é a nota honesta: a validação cruzada escolheu o modelo, o teste diz como funciona com dados novos.

In [8]:
best_model = model.best_estimator_
print("Gradient Boosting ajustado")
print("Exactitud en entrenamiento:", round(best_model.score(X_train_sel, y_train), 3))
print("Exactitud en test:", round(best_model.score(X_test_sel, y_test), 3))

Gradient Boosting ajustado
Exactitud en entrenamiento: 0.825
Exactitud en test: 0.79


**➜ Qué nos llevamos:** el Grid Search elige árboles un poco más profundos (4), más árboles (200) y pasos más pequeños (0,05). La exactitud en test sube de 78,4 % a 79,0 %: menos de un punto. Los valores por defecto ya eran buenos; el ajuste afina, no hace milagros.

> **➜ O que tiramos:** o Grid Search escolhe árvores um pouco mais profundas (4), mais árvores (200) e passos mais pequenos (0,05). A exactidão no teste sobe de 78,4 % para 79,0 %: menos de um ponto. Os valores por defeito já eram bons; o ajuste afina, não faz milagres.

### Random Search con el Random Forest

La lección también enseña **Random Search**: en vez de probar todas las combinaciones, prueba `n_iter` al azar. Lo usamos con el segundo mejor modelo del lab de ensembles, el Random Forest (78,4 % en test con 100 árboles y profundidad 20).

> **Random Search com o Random Forest (português).** Em vez de provar todas as combinações, prova `n_iter` ao acaso. Usamo-lo com o segundo melhor modelo do lab de ensembles, o Random Forest.

**Cómo funciona**

1. Define los valores posibles: `n_estimators` (100 a 500), `max_depth` (5 a 20 o sin límite) y `max_leaf_nodes` (máximo de hojas por árbol), como en la lección.
2. `RandomizedSearchCV` con `n_iter=10`: solo 10 de las 100 combinaciones posibles, cada una con `cv=5`.
3. `random_state=0` para que el sorteo de combinaciones sea siempre el mismo.

**Por qué así:** con muchas combinaciones posibles, probarlas todas tardaría mucho; 10 al azar suelen acercarse a la mejor en una fracción del tiempo.

**Herramientas:** `RandomizedSearchCV(estimator=..., param_distributions=..., n_iter=10, cv=5)` (lección 7.4).

> **Como funciona (português)**
>
> 1. Define os valores possíveis: `n_estimators` (100 a 500), `max_depth` (5 a 20 ou sem limite) e `max_leaf_nodes` (máximo de folhas por árvore), como na aula.
> 2. `RandomizedSearchCV` com `n_iter=10`: só 10 das 100 combinações possíveis, cada uma com `cv=5`.
> 3. `random_state=0` para o sorteio das combinações ser sempre o mesmo.
>
> **Porquê assim:** com muitas combinações possíveis, prová-las todas demoraria muito; 10 ao acaso costumam aproximar-se da melhor numa fracção do tempo.

In [9]:
grid_rf = {"n_estimators": [100, 200, 300, 500],
           "max_depth": [5, 10, 15, 20, None],
           "max_leaf_nodes": [50, 100, 250, 500, None]}

rf = RandomForestClassifier(random_state=0)
model_rf = RandomizedSearchCV(estimator=rf, param_distributions=grid_rf, n_iter=10, cv=5, random_state=0)
model_rf.fit(X_train_sel, y_train)

best_rf = model_rf.best_estimator_
print("Mejores hiperparámetros:", model_rf.best_params_)
print("Exactitud media en validación cruzada:", round(model_rf.best_score_, 3))
print("Exactitud en test:", round(best_rf.score(X_test_sel, y_test), 3))

Mejores hiperparámetros: {'n_estimators': 300, 'max_leaf_nodes': 100, 'max_depth': None}
Exactitud media en validación cruzada: 0.795
Exactitud en test: 0.785


### Comparación final

> **Comparação final (português).**

In [10]:
comparacion = pd.DataFrame({
    "modelo": ["Gradient Boosting sin ajustar", "Gradient Boosting + Grid Search", "Random Forest + Random Search"],
    "entrenamiento": [gb_base.score(X_train_sel, y_train), best_model.score(X_train_sel, y_train), best_rf.score(X_train_sel, y_train)],
    "test": [gb_base.score(X_test_sel, y_test), best_model.score(X_test_sel, y_test), best_rf.score(X_test_sel, y_test)]})
comparacion.round(3).set_index("modelo")

,entrenamiento,test
modelo,,
Gradient Boosting sin ajustar,0.813,0.784
Gradient Boosting + Grid Search,0.825,0.790
Random Forest + Random Search,0.843,0.785


**➜ Qué nos llevamos:** el mejor modelo del lab es el **Gradient Boosting ajustado con Grid Search: 79 % de exactitud en test**. El Random Search encuentra un Random Forest parecido en la validación, pero algo peor en test. Lo que aprendemos: los hiperparámetros se eligen con validación cruzada sobre el entrenamiento y el test solo se usa una vez, al final; y el ajuste da mejoras pequeñas cuando el modelo ya era bueno. Que sean los mejores de los que probamos no garantiza que sean los mejores posibles.

> **➜ O que tiramos:** o melhor modelo do lab é o **Gradient Boosting ajustado com Grid Search: 79 % de exactidão no teste**. O Random Search encontra um Random Forest parecido na validação, mas um pouco pior no teste. O que aprendemos: os hiperparâmetros escolhem-se com validação cruzada sobre o treino e o teste só se usa uma vez, no fim; e o ajuste dá melhorias pequenas quando o modelo já era bom. Serem os melhores dos que provámos não garante que sejam os melhores possíveis.